# Fine-tune mBART for Sinhala Language Error Correction
## Kaggle Training Notebook - Optimized for 5% Data Usage
This notebook fine-tunes a multilingual BERT (mBART) model for Sinhala sentence error correction using Kaggle datasets.

**Data Setup**: Uses noisy.txt (Input) and original.txt (Target) - 5% of data for training

## 1. Install Required Libraries

In [1]:
# Install required packages
import subprocess
import sys

packages = [
    'nltk',
    'transformers[torch]',
    'rouge_score',
    'evaluate',
    'jiwer',
    'sacrebleu',
    'datasets'
]

print("Installing required packages...")
for package in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

print("\n✓ All packages installed successfully!")

Installing required packages...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 38.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.1/104.1 kB 4.4 MB/s eta 0:00:00

✓ All packages installed successfully!


## 2. Import Libraries and Configure Device

In [2]:
import json
import os
import time
import random
import re
import string
import numpy as np
import pandas as pd
import torch
import jiwer
from datasets import Dataset
from evaluate import load
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from sklearn.model_selection import train_test_split
from transformers import (
    AutoModelForSeq2SeqLM,
    Seq2SeqTrainingArguments,
    AutoTokenizer,
    EarlyStoppingCallback,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    GenerationConfig,
    pipeline
)
from transformers.trainer_utils import get_last_checkpoint
import nltk

nltk.download('punkt')

# Configure device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✓ Using device: {device}")

if torch.cuda.is_available():
    print(f"✓ GPU: {torch.cuda.get_device_name(0)}")
    print(f"✓ GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

# Configure output directory for Kaggle
output_dir = "/kaggle/working/sinhala_model"
os.makedirs(output_dir, exist_ok=True)
print(f"\n✓ Output directory: {output_dir}")

2026-01-09 19:28:01.904886: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1767986882.085740      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1767986882.138454      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1767986882.572578      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767986882.572616      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767986882.572619      24 computation_placer.cc:177] computation placer alr

✓ Using device: cuda
✓ GPU: Tesla T4
✓ GPU Memory: 15.83 GB

✓ Output directory: /kaggle/working/sinhala_model


[nltk_data] Downloading package punkt to /usr/share/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


## 3. Load Sinhala Dataset from CSV Files

In [3]:
# Load Sinhala dataset from Kaggle input directory
data_dir = "/kaggle/input"
train_csv_path = None
test_csv_path = None

# Find CSV files in the input directory
print("Searching for CSV files...\n")
for root, dirs, files in os.walk(data_dir):
    for file in files:
        if file.endswith('.csv'):
            file_path = os.path.join(root, file)
            print(f"  Found: {file_path}")
            if 'train' in file.lower():
                train_csv_path = file_path
            elif 'test' in file.lower():
                test_csv_path = file_path

print(f"\nTrain CSV path: {train_csv_path}")
print(f"Test CSV path: {test_csv_path}")

# Load datasets
train_df = None
test_df = None

if train_csv_path:
    train_df = pd.read_csv(train_csv_path, encoding='utf-8')
    print(f"\n✓ Train Dataset loaded:")
    print(f"  - Shape: {train_df.shape}")
    print(f"  - Columns: {train_df.columns.tolist()}")
    print(f"\n  Sample rows:")
    for i in range(min(3, len(train_df))):
        print(f"    {i+1}. Input: {train_df.iloc[i]['Input'][:50]}...")
        print(f"       Target: {train_df.iloc[i]['Target'][:50]}...\n")
else:
    print("\n✗ ERROR: Could not find train.csv")

if test_csv_path:
    test_df = pd.read_csv(test_csv_path, encoding='utf-8')
    print(f"\n✓ Test Dataset loaded:")
    print(f"  - Shape: {test_df.shape}")
    print(f"  - Columns: {test_df.columns.tolist()}")
else:
    print("\nℹ  No test.csv found - will split training data")

Searching for CSV files...

  Found: /kaggle/input/sc-dataset/train.csv
  Found: /kaggle/input/sc-dataset/test.csv

Train CSV path: /kaggle/input/sc-dataset/train.csv
Test CSV path: /kaggle/input/sc-dataset/test.csv

✓ Train Dataset loaded:
  - Shape: (1629, 2)
  - Columns: ['Input', 'Target']

  Sample rows:
    1. Input: මෙහදී පොලිස්පති පූජිත් ජයසූන්දර ලබාදුන සාක්‍ෂි ඉත ...
       Target: මෙහිදී පොලිස්පති පූජිත් ජයසුන්දර ලබාදුන් සාක්‍ෂි ඉ...

    2. Input: මේ ඳෙකටම අමතරව කළු බළල්ල කහ ඹළල්ලු වැනි විවිධ නම්ව...
       Target: මේ දෙකටම අමතරව කළු බළල්ලු කහ බළල්ලු වැනි විවිධ නම්...

    3. Input: රංගන කාර්යයට වැඩිය නිර්මාණාත්මක පැත්ගතට යොමු වුණේ ...
       Target: රංගන කාර්යයට වැඩිය නිර්මාණාත්මක පැත්තට යොමු වුණේ එ...


✓ Test Dataset loaded:
  - Shape: (408, 2)
  - Columns: ['Input', 'Target']


## 4. Sinhala Error Generation Functions
Sinhala-specific error simulation functions for training data augmentation

In [4]:
# Sinhala character confusions and homophones
# Based on SINHALA_CHARACTER_REFERENCE.md

def replace_with_sinhala_homophones(word):
    """Replace characters with similar-looking Sinhala characters"""
    sinhala_homophones = {
        # Vowels (ස්වර)
        'අ': 'ආ', 'ආ': 'අ',
        'ඉ': 'ඊ', 'ඊ': 'ඉ',
        'උ': 'ඌ', 'ඌ': 'උ',
        'එ': 'ඒ', 'ඒ': 'එ',
        'ඔ': 'ඕ', 'ඕ': 'ඔ',
        # Consonants that are easily confused
        'ක': 'ග', 'ග': 'ක',
        'ච': 'ජ', 'ජ': 'ච',
        'ට': 'ඩ', 'ඩ': 'ට',
        'ත': 'ද', 'ද': 'ත',
        'ප': 'බ', 'බ': 'ප',
        'ය': 'ල', 'ල': 'ය',
        'ස': 'ශ', 'ශ': 'ස',
    }
    
    if len(word) == 0:
        return word
    
    idx = random.randint(0, len(word) - 1)
    char = word[idx]
    
    if char in sinhala_homophones:
        return word[:idx] + sinhala_homophones[char] + word[idx + 1:]
    return word


def swap_adjacent_chars(word):
    """Swap adjacent characters"""
    if len(word) < 2:
        return word
    idx = random.randint(0, len(word) - 2)
    return word[:idx] + word[idx + 1] + word[idx] + word[idx + 2:]


def remove_char(word):
    """Remove a random character"""
    if len(word) < 2:
        return word
    idx = random.randint(0, len(word) - 1)
    return word[:idx] + word[idx + 1:]


def insert_char(word):
    """Insert a random Sinhala character"""
    sinhala_chars = 'අආඉඊඋඌඑඒඔඕකඛගඝචඡජඣටඨඩඪණඬતતුදධනඳපඳබභමඹයරලවශෂසහ'
    idx = random.randint(0, len(word))
    char = random.choice(sinhala_chars)
    return word[:idx] + char + word[idx:]


def transpose_char(word):
    """Transpose (swap) adjacent characters"""
    if len(word) < 2:
        return word
    idx = random.randint(0, len(word) - 2)
    return word[:idx] + word[idx + 1] + word[idx] + word[idx + 2:]


def repeat_char(word):
    """Repeat a character"""
    if len(word) < 1:
        return word
    idx = random.randint(0, len(word) - 1)
    return word[:idx] + word[idx] + word[idx] + word[idx + 1:]


def remove_diacritic(word):
    """Remove diacritical marks (යුක්තිය)"""
    sinhala_diacritics = 'ාැෙිීුූෘෝෞ'
    new_word = ''
    for char in word:
        if char in sinhala_diacritics and random.random() < 0.5:
            continue
        new_word += char
    return new_word if new_word else word


def replace_wrong_diacritic(word):
    """Replace correct diacritics with wrong ones"""
    diacritic_replacements = {
        'ා': 'ැ', 'ැ': 'ෙ', 'ෙ': 'ි',
        'ි': 'ු', 'ු': 'ූ',
    }
    
    new_word = ''
    for char in word:
        if char in diacritic_replacements and random.random() < 0.5:
            new_word += diacritic_replacements[char]
        else:
            new_word += char
    return new_word


def modify_word_based_on_error_type(word, error_type):
    """Apply specific error type to word"""
    if error_type == "swap":
        return swap_adjacent_chars(word)
    elif error_type == "remove":
        return remove_char(word)
    elif error_type == "insert":
        return insert_char(word)
    elif error_type == "homophone":
        return replace_with_sinhala_homophones(word)
    elif error_type == "transpose":
        return transpose_char(word)
    elif error_type == "repeat":
        return repeat_char(word)
    elif error_type == "remove_diacritic":
        return remove_diacritic(word)
    elif error_type == "replace_diacritic":
        return replace_wrong_diacritic(word)
    return word


def introduce_errors(text, error_rate):
    """Introduce random errors in Sinhala text"""
    words = text.split()
    if len(words) == 0:
        return text
    
    num_errors = random.randint(0, 1)
    for _ in range(num_errors):
        if random.random() < error_rate:
            idx = random.randint(0, len(words) - 1)
            error_types = ["swap", "remove", "insert", "homophone", 
                          "transpose", "repeat", "remove_diacritic", "replace_diacritic"]
            error_type = random.choice(error_types)
            words[idx] = modify_word_based_on_error_type(words[idx], error_type)
    
    return " ".join(words)


print("✓ Sinhala error generation functions loaded")

✓ Sinhala error generation functions loaded


## 5. Prepare Training Data

In [5]:
# Configuration
SINHALA_LANG_CODE = "si_LK"  # Sinhala, Sri Lanka
MODEL_NAME = "facebook/mbart-large-50"
MAX_TOKEN_LENGTH = 32
BATCH_SIZE = 8  # Reduced for 5% data
NUM_EPOCHS = 5  # Increased for small dataset
LEARNING_RATE = 5e-5
WARMUP_STEPS = 100
ERROR_RATE = 0.3  # For augmentation
DATA_PERCENTAGE = 50  # ========== CHANGE THIS VALUE (1-100) ==========

print("=" * 60)
print("TRAINING CONFIGURATION")
print("=" * 60)
print(f"Language Code: {SINHALA_LANG_CODE}")
print(f"Base Model: {MODEL_NAME}")
print(f"Max Token Length: {MAX_TOKEN_LENGTH}")
print(f"Batch Size: {BATCH_SIZE}")
print(f"Num Epochs: {NUM_EPOCHS}")
print(f"Learning Rate: {LEARNING_RATE}")
print(f"Data Percentage: {DATA_PERCENTAGE}%")
print("=" * 60)

TRAINING CONFIGURATION
Language Code: si_LK
Base Model: facebook/mbart-large-50
Max Token Length: 32
Batch Size: 8
Num Epochs: 5
Learning Rate: 5e-05
Data Percentage: 50%


In [6]:
# Prepare data pairs from CSV
if train_df is not None:
    print("\nPreparing training data...")
    
    # Extract Input and Target columns
    input_texts = train_df['Input'].astype(str).tolist()
    target_texts = train_df['Target'].astype(str).tolist()
    
    # Create training pairs
    train_pairs = []
    for input_text, target_text in zip(input_texts, target_texts):
        if len(input_text.strip()) > 0 and len(target_text.strip()) > 0:
            train_pairs.append((input_text.strip(), target_text.strip()))
    
    print(f"✓ Total training pairs available: {len(train_pairs)}")
    
    # Apply percentage filter
    percentage_count = max(1, int(len(train_pairs) * (DATA_PERCENTAGE / 100)))
    train_pairs = train_pairs[:percentage_count]
    print(f"✓ Using {DATA_PERCENTAGE}% of data: {len(train_pairs)} samples")
    
    # Split into train and validation
    train_data, val_data = train_test_split(
        train_pairs,
        test_size=0.1,
        random_state=42
    )
    
    print(f"✓ Train split: {len(train_data)} samples")
    print(f"✓ Validation split: {len(val_data)} samples")
    
    # Show samples
    print(f"\n--- Sample Training Data ---")
    for i in range(min(3, len(train_data))):
        print(f"\nSample {i+1}:")
        print(f"  Input (noisy): {train_data[i][0]}")
        print(f"  Target (clean): {train_data[i][1]}")
else:
    print("✗ No training data loaded!")


Preparing training data...
✓ Total training pairs available: 1629
✓ Using 50% of data: 814 samples
✓ Train split: 732 samples
✓ Validation split: 82 samples

--- Sample Training Data ---

Sample 1:
  Input (noisy): මේ දෙවල් අදඨ වැඩියෙන් කරන්න ඕනේ .
  Target (clean): මේ දේවල් අදට වැඩියෙන් කරන්න ඕනේ .

Sample 2:
  Input (noisy): සේවකයන් කඞියන් මෙන් එහෙ මෙහෙ යමින් යුහුසුළුව සෙවාදායකයනට උපරිමව සේවය සැපයීමේ නීරතව සිටයහ .
  Target (clean): සේවකයන් කඩියන් මෙන් එහෙ මෙහෙ යමින් යුහුසුලුව සේවාදායකයනට උපරිමව සේවය සැපයීමේ නිරතව සිටියහ .

Sample 3:
  Input (noisy): එය තමයි අපේ වගකිම වනු අැත්තේ .
  Target (clean): එය තමයි අපේ වගකීම වනු ඇත්තේ .


## 6. Load mBART Tokenizer

In [7]:
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Set language code
tokenizer.src_lang = (SINHALA_LANG_CODE)
tokenizer.tgt_lang = SINHALA_LANG_CODE

print(f"✓ Tokenizer loaded: {type(tokenizer)}")
print(f"✓ Vocab size: {len(tokenizer)}")

# Test tokenizer with sample Sinhala text
sample_text = "සිංහල භාෂාව උතුරු සිරිපත් දේශයේ"
print(f"\nTokenizer test:")
print(f"  Input: {sample_text}")
tokens = tokenizer.tokenize(sample_text)
print(f"  Tokens: {tokens}")
print(f"  Token count: {len(tokens)}")

Loading tokenizer...


tokenizer_config.json:   0%|          | 0.00/531 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/649 [00:00<?, ?B/s]

✓ Tokenizer loaded: <class 'transformers.models.mbart50.tokenization_mbart50_fast.MBart50TokenizerFast'>
✓ Vocab size: 250054

Tokenizer test:
  Input: සිංහල භාෂාව උතුරු සිරිපත් දේශයේ
  Tokens: ['▁සිංහල', '▁භාෂාව', '▁උතුරු', '▁සිරි', 'පත්', '▁දේශ', 'යේ']
  Token count: 7


## 7. Create Datasets

In [8]:
class SinhalaDataset(torch.utils.data.Dataset):
    """Custom dataset for Sinhala error correction"""
    def __init__(self, data, tokenizer, max_length=32):
        self.data = data
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        input_text, target_text = self.data[idx]
        
        input_encodings = self.tokenizer(
            input_text, 
            return_tensors='pt', 
            max_length=self.max_length, 
            padding='max_length', 
            truncation=True
        )

        target_encodings = self.tokenizer(
            target_text, 
            return_tensors='pt', 
            max_length=self.max_length, 
            padding='max_length', 
            truncation=True
        )

        return {
            "input_ids": input_encodings['input_ids'].squeeze(),
            "attention_mask": input_encodings["attention_mask"].squeeze(),
            "labels": target_encodings['input_ids'].squeeze()
        }


# Create dataset objects
if 'train_data' in locals() and len(train_data) > 0:
    print("Creating dataset objects...\n")
    
    train_dataset = SinhalaDataset(train_data, tokenizer, max_length=MAX_TOKEN_LENGTH)
    val_dataset = SinhalaDataset(val_data, tokenizer, max_length=MAX_TOKEN_LENGTH)
    
    print(f"✓ Train dataset size: {len(train_dataset)}")
    print(f"✓ Validation dataset size: {len(val_dataset)}")
    
    # Show sample
    sample = train_dataset[0]
    print(f"\nSample from training dataset:")
    print(f"  Input IDs shape: {sample['input_ids'].shape}")
    print(f"  Input text: {tokenizer.decode(sample['input_ids'])}")
    print(f"  Target text: {tokenizer.decode(sample['labels'])}")

Creating dataset objects...

✓ Train dataset size: 732
✓ Validation dataset size: 82

Sample from training dataset:
  Input IDs shape: torch.Size([32])
  Input text: si_LK මේ දෙවල් අදඨ වැඩියෙන් කරන්න ඕනේ .</s><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad>
  Target text: si_LK මේ දේවල් අදට වැඩියෙන් කරන්න ඕනේ .</s><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad>


## 8. Load mBART Model

In [9]:
print("Loading mBART model...")
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

print(f"✓ Model loaded: {type(model)}")
print(f"✓ Model parameters: {sum(p.numel() for p in model.parameters())/1e6:.2f}M")

# Check for existing checkpoint
last_checkpoint = get_last_checkpoint(output_dir)
if last_checkpoint:
    print(f"✓ Found checkpoint: {last_checkpoint}")
    model = AutoModelForSeq2SeqLM.from_pretrained(last_checkpoint)
else:
    print("ℹ  No checkpoint found - starting fresh training")

# Ensure generation starts in target language (Sinhala)
model.config.forced_bos_token_id = tokenizer.lang_code_to_id[SINHALA_LANG_CODE]

Loading mBART model...


pytorch_model.bin:   0%|          | 0.00/2.44G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.44G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/261 [00:00<?, ?B/s]

✓ Model loaded: <class 'transformers.models.mbart.modeling_mbart.MBartForConditionalGeneration'>
✓ Model parameters: 610.88M
ℹ  No checkpoint found - starting fresh training


## 9. Set Up Training Arguments

In [10]:
# Data collator
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

# Training arguments
training_args = Seq2SeqTrainingArguments(
    output_dir=output_dir,
    overwrite_output_dir=True,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    warmup_steps=WARMUP_STEPS,
    weight_decay=0.01,
    learning_rate=LEARNING_RATE,
    logging_dir='/kaggle/working/logs',
    logging_steps=10,
    save_steps=100,
    eval_steps=100,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    report_to='tensorboard',
    bf16=torch.cuda.is_available(),
    gradient_accumulation_steps=2,
    # Ensure eval/save strategies match for load_best_model_at_end
    eval_strategy='steps',
    save_strategy='steps',
    
)

print("\n" + "=" * 60)
print("TRAINING ARGUMENTS")
print("=" * 60)
print(f"Output directory: {training_args.output_dir}")
print(f"Epochs: {training_args.num_train_epochs}")
print(f"Batch size (train): {training_args.per_device_train_batch_size}")
print(f"Batch size (eval): {training_args.per_device_eval_batch_size}")
print(f"Learning rate: {training_args.learning_rate}")
print(f"Mixed precision: {training_args.bf16}")
print("=" * 60)


TRAINING ARGUMENTS
Output directory: /kaggle/working/sinhala_model
Epochs: 5
Batch size (train): 8
Batch size (eval): 8
Learning rate: 5e-05
Mixed precision: True


## 10. Load Evaluation Metrics

In [11]:
print("Loading evaluation metrics...")
cer_metric = load("cer")
wer_metric = load("wer")
bleu_metric = load("sacrebleu")
rouge_metric = load('rouge')

print("✓ Metrics loaded")

def compute_metrics(eval_preds):
    """Compute evaluation metrics"""
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]

    # If logits provided, take argmax to get token ids
    try:
        if hasattr(preds, 'ndim') and preds.ndim == 3:
            preds = np.argmax(preds, axis=-1)
    except Exception:
        pass

    # Replace -100 (masked) tokens with pad token
    preds = np.where(preds != -100, preds, tokenizer.pad_token_id)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    
    # Decode predictions and labels
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    result = {}
    
    # ROUGE score
    try:
        _rouge = rouge_metric.compute(predictions=decoded_preds, references=decoded_labels)
        result['rouge1'] = _rouge['rouge1']
        result['rougeL'] = _rouge['rougeL']
    except Exception:
        pass
    
    # CER and WER
    try:
        _cer = cer_metric.compute(predictions=decoded_preds, references=decoded_labels)
        result['cer'] = _cer
    except Exception:
        pass
    
    try:
        _wer = wer_metric.compute(predictions=decoded_preds, references=decoded_labels)
        result['wer'] = _wer
    except Exception:
        pass
    
    return result

Loading evaluation metrics...


✓ Metrics loaded


## 11. Train the Model

In [12]:
print("\n" + "=" * 60)
print("STARTING TRAINING")
print("=" * 60)
print(f"Start time: {time.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Total training steps: {len(train_dataset) // BATCH_SIZE * NUM_EPOCHS}")
print("=" * 60 + "\n")

# Create trainer
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(
        early_stopping_patience=3,
        early_stopping_threshold=0.001
    )]
)

# Train
train_result = trainer.train(resume_from_checkpoint=last_checkpoint)

print("\n" + "=" * 60)
print("✓ TRAINING COMPLETED")
print("=" * 60)
print(f"Final loss: {train_result.training_loss:.4f}")
train_runtime = train_result.metrics.get('train_runtime')
if train_runtime is not None:
    print(f"Training time: {train_runtime / 3600:.2f} hours")
else:
    print("Training time: N/A")


STARTING TRAINING
Start time: 2026-01-09 19:28:45
Total training steps: 455



/tmp/ipykernel_24/2278327773.py:9: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(
/usr/local/lib/python3.12/dist-packages/transformers/data/data_collator.py:740: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  batch["labels"] = torch.tensor(batch["labels"], dtype=torch.int64)
/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Step,Training Loss,Validation Loss,Rouge1,Rougel,Cer,Wer
100,1.625300,1.732081,0.000000,0.000000,0.292707,0.454248


/usr/local/lib/python3.12/dist-packages/transformers/modeling_utils.py:3918: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 200, 'early_stopping': True, 'num_beams': 5, 'forced_bos_token_id': 250022}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
There were missing keys in the checkpoint model loaded: ['model.encoder.embed_tokens.weight', 'model.decoder.embed_tokens.weight', 'lm_head.weight'].



✓ TRAINING COMPLETED
Final loss: 4.3375
Training time: 0.07 hours


## 12. Save Model and Evaluate

In [13]:
# Save model
print("Saving model...")
model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)
print(f"✓ Model saved to: {output_dir}")

# Evaluate
print("\nEvaluating model...")
eval_results = trainer.evaluate(eval_dataset=val_dataset)

print("\n" + "=" * 60)
print("EVALUATION RESULTS")
print("=" * 60)
for key, value in eval_results.items():
    if isinstance(value, float):
        print(f"{key}: {value:.4f}")
    else:
        print(f"{key}: {value}")

# Save results to file
results_file = os.path.join(output_dir, 'training_results.txt')
with open(results_file, 'w', encoding='utf-8') as f:
    f.write("=" * 60 + "\n")
    f.write("SINHALA LANGUAGE ERROR CORRECTION - FINE-TUNING RESULTS\n")
    f.write("=" * 60 + "\n\n")
    
    f.write("CONFIGURATION\n")
    f.write("-" * 60 + "\n")
    f.write(f"Language Code: {SINHALA_LANG_CODE}\n")
    f.write(f"Base Model: {MODEL_NAME}\n")
    f.write(f"Training Epochs: {NUM_EPOCHS}\n")
    f.write(f"Batch Size: {BATCH_SIZE}\n")
    f.write(f"Learning Rate: {LEARNING_RATE}\n")
    f.write(f"Max Token Length: {MAX_TOKEN_LENGTH}\n\n")
    
    f.write("TRAINING RESULTS\n")
    f.write("-" * 60 + "\n")
    f.write(f"Final Training Loss: {train_result.training_loss:.4f}\n")
    train_runtime = train_result.metrics.get('train_runtime')
    if train_runtime is not None:
        f.write(f"Training Time: {train_runtime / 3600:.2f} hours\n\n")
    else:
        f.write("Training Time: N/A\n\n")
    
    f.write("EVALUATION METRICS\n")
    f.write("-" * 60 + "\n")
    for key, value in eval_results.items():
        if isinstance(value, float):
            f.write(f"{key}: {value:.4f}\n")
        else:
            f.write(f"{key}: {value}\n")

print(f"\n✓ Results saved to: {results_file}")

Saving model...
✓ Model saved to: /kaggle/working/sinhala_model

Evaluating model...


/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(



EVALUATION RESULTS
eval_loss: 1.7321
eval_rouge1: 0.0000
eval_rougeL: 0.0000
eval_cer: 0.2927
eval_wer: 0.4542
eval_runtime: 4.9851
eval_samples_per_second: 16.4490
eval_steps_per_second: 1.2040
epoch: 5.0000

✓ Results saved to: /kaggle/working/sinhala_model/training_results.txt


## 13. Test Model with Examples

In [14]:
# Test with some validation samples
print("\n" + "=" * 60)
print("TESTING MODEL WITH VALIDATION SAMPLES")
print("=" * 60 + "\n")

pipe = pipeline('text2text-generation', model=model, tokenizer=tokenizer, device=0 if torch.cuda.is_available() else -1)

# Test on random validation samples
num_tests = min(5, len(val_data))
for i in range(num_tests):
    input_text, target_text = val_data[i]
    
    # Generate correction
    result = pipe(input_text, max_length=32, num_beams=4)
    predicted_text = result[0]['generated_text']
    
    print(f"Test {i+1}:")
    print(f"  Input (Noisy):     {input_text}")
    print(f"  Predicted (Clean): {predicted_text}")
    print(f"  Target (Clean):    {target_text}")
    print()

Device set to use cuda:0



TESTING MODEL WITH VALIDATION SAMPLES

Test 1:
  Input (Noisy):     සාමාන්‍ය පෙළ විභාඟය සඳහ හැදුනුම්පත් අයදුම් කර ඇථි බොහෝ දෙනෙකුට මේ වන විට මෙම ෂහතික සකස කරමින් ථිබෙන බවත් තවත කට පමණ ඒවපා නිකුත් කිරීමට නියමිතව තිබේ .
  Predicted (Clean): සාමාන් ය පෙළ විභාගයට පෙනී සිටින අයදුම්කරුවන් බොහෝ දෙනෙකුට මේ වන විට මෙම සහතික පත් සකස් කරමින් නිකුත් කිරීමට නියමිතව තිබේ .
  Target (Clean):    සාමාන්‍ය පෙළ විභාගය සඳහා හැඳුනුම්පත් අයදුම් කර ඇති බොහෝ දෙනෙකුට මේ වන විට මෙම සහතික සකස් කරමින් තිබෙන බවත් තවත් කට පමණ ඒවා නිකුත් කිරීමට නියමිතව තිබේ .

Test 2:
  Input (Noisy):     කුඬු ඹිස්නස් කරනවා කොයි තරම් ඇල්ලූවත් ඒවා ව්‍යාප්ත වෙනවා .
  Predicted (Clean): කුඩේ පාස්නස් කරනවා කොයි තරම් ඇල්ලූවත් ඒවා ව් යාප්ත වෙනවා .
  Target (Clean):    කුඩු බිස්නස් කරනවා කොයි තරම් ඇල්ලුවත් ඒවා ව්‍යාප්ත වෙනවා .

Test 3:
  Input (Noisy):     රාජ්‍ය හා රාජ්‍ය නොවන විශ්වවිද්‍යාලවලට ඇතළුවන සීසුන් බහුතරය නිර්මාණය කෙරඖෙන්නේ මේ අතරින් පාසල් හැත්තෑවකිනි .
  Predicted (Clean): රාජ් ය හා පෞද්ගලික විශ්වවිද් යාලවලට ඇතුළුවන සිසුන් බහු